# Across-night HR and quiet-period HRV

One subject, all processed night folders. SPT includes the entire lights-off–wake-up crop, not independently confirmed sleep. Bounds are estimated from saved acceleration timestamps plus one 100-Hz sample; partial final minutes are omitted. All calculations use UTC after interpreting naive saved timestamps in Europe/Rome.

**Main view:** non-overlapping 60-second windows anchored to SPT start. **Companion:** 300-second windows every 60 seconds, centred at the same minute centres. They overlap and are not independent observations. Fixed five-minute windows must fit wholly within SPT and one quiet segment. No SDNN is calculated.

HR is FIT-recorded HR, sampled to 1 Hz with no extrapolation and no interpolation across gaps >3 s; at least 90% of each minute must be valid. Whole-SPT and quiet-period HR are separate. RMSSD uses saved GP-cleaned BBIs only inside a single saved quiet segment and delivery run; timestamps are callback arrival proxies, not exact beat times. lnRMSSD is ln(RMSSD in ms); zero RMSSD has undefined log.

Quiet periods inherit the export settings, including the minimum movement duration and post-movement guard. The original exports typically ignore bursts <2 s; thus “quiet” is the pipeline definition, not proof of complete immobility. The provenance table below exposes these settings. We do not concatenate separated quiet periods or interpolate missing minute values.

The default HRV limit is at most 5% interpolated intervals (an adjustable analysis choice, stricter than the legacy export). Windows also require >=30 intervals/minute, finite positive BBIs, continuous sequence numbers, callback/edge gaps <=5 s, and total BBI duration within 90–110% of window length. All rejected windows retain reasons.

One-minute HRV is exploratory and should be compared with the five-minute profile, not assumed interchangeable. Across-night means give each contributing night equal weight; shaded bands show between-night SD, not independent-subject uncertainty. No inferential p-values are calculated.

In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
MODULE_DIR = next(p for base in [Path.cwd(), *Path.cwd().parents] for p in [base, base / "offline_processing"] if (p / "nightly_hr_hrv.py").exists())
sys.path.insert(0, str(MODULE_DIR))
from nightly_hr_hrv import analyze_subject, aligned_profiles
ROOT = Path(os.environ.get("OVERNIGHT_PROCESSED_ROOT", str(Path.home() / "Documents/garmin/sleep/garmin_processed")))
NAIVE_TIMEZONE = "Europe/Rome"
MAX_INTERPOLATED_FRACTION = 0.05
OUTPUT_DIR = MODULE_DIR / "outputs" / "nightly_hr_hrv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
LABELS = {"hr_mean_bpm": "Whole-SPT HR (bpm)", "quiet_hr_bpm": "Quiet-period HR (bpm)", "rmssd_ms": "Quiet RMSSD (ms)", "lnrmssd": "Quiet lnRMSSD"}
def finish(name):
    plt.gcf().savefig(OUTPUT_DIR / (name + ".png"), dpi=140, bbox_inches="tight")
    plt.show()


In [ ]:
result = analyze_subject(ROOT, naive_timezone=NAIVE_TIMEZONE, max_interpolated_fraction=MAX_INTERPOLATED_FRACTION)
minutes, five = result["minutes"], result["five_minutes"]
nights, summary = result["nights"], result["summary"]
print(f"{len(nights)} nights; {minutes.rmssd_ms.count()} valid one-minute HRV windows; {five.rmssd_ms.count()} valid five-minute windows")
display(nights)
display(minutes.loc[~minutes.included].groupby("exclusion_reason").size().rename("Rejected one-minute windows"))
display(five.loc[~five.included].groupby("exclusion_reason").size().rename("Rejected five-minute windows"))


## Nightly summaries and chronological trends

Summaries use the non-overlapping one-minute values, never the overlapping five-minute windows. Mean minute-wise RMSSD is not whole-night RMSSD. HRV summaries describe observed quiet minutes and can change when quiet-period coverage changes. The table includes means, medians, quartiles, SD, and valid-minute counts. Across-night means below weight nights equally. Dates label recording-folder start dates, including after-midnight starts.

In [ ]:
display(summary)
display(summary[[m + "_mean" for m in LABELS]].agg(["mean", "std", "count"]))
x = pd.to_datetime(summary.index, format="%Y-%m-%d-%H-%M-%S")
fig, axes = plt.subplots(3, 2, figsize=(14, 11), constrained_layout=True)
for ax, (metric, label) in zip(axes.flat, LABELS.items()):
    mean = summary[metric + "_mean"]
    ax.plot(x, mean, "o-", label="Nightly mean")
    ax.plot(x, summary[metric + "_median"], "s--", alpha=.7, label="Nightly median")
    ax.axhline(mean.mean(), color="gray", linestyle=":", label="Equal-night mean")
    ax.set_ylabel(label)
axes[0, 0].legend(fontsize=8)
axes[2, 0].plot(x, nights.loc[summary.index, "rmssd_spt_coverage_pct"], "o-", color="#167D8D")
axes[2, 0].set_ylabel("Valid HRV minutes / SPT (%)")
axes[2, 1].plot(x, nights.loc[summary.index, "spt_minutes"] / 60, "o-", color="#A6761D")
axes[2, 1].set_ylabel("SPT duration (hours)")
for ax in axes.flat: ax.tick_params(axis="x", rotation=45)
finish("nightly_trends")


## One-minute heatmaps

Each row is a night. Grey means rejected/unavailable data or time beyond that night's SPT; it is not zero. Colour limits use the 2nd–98th percentiles for visibility; extreme values saturate the scale. These show actual minutes from SPT start, not stretched nights.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 15), constrained_layout=True)
for ax, (metric, label) in zip(axes, LABELS.items()):
    matrix = minutes.pivot(index="night", columns="minute", values=metric).reindex(nights.index)
    values = matrix.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    lo, hi = np.quantile(finite, [.02, .98]) if len(finite) else (0, 1)
    cmap = plt.get_cmap("viridis").copy(); cmap.set_bad("#dedede")
    image = ax.imshow(values, aspect="auto", interpolation="nearest", cmap=cmap, vmin=lo, vmax=hi,
                      extent=[0, matrix.shape[1], len(matrix), 0])
    ax.set_yticks(np.arange(len(matrix)) + .5, [n[:10] for n in matrix.index], fontsize=7)
    ax.set(xlabel="Minutes from SPT start", title=label)
    fig.colorbar(image, ax=ax, pad=.01, extend="both")
finish("minute_heatmaps")


## Aligned across-night profiles

Thin curves show individual nights, thick curves the equal-night mean, and shading ±1 between-night SD. A lower panel shows contributing nights separately for each metric. Late portions can contain fewer nights. No gaps are filled. Relative-SPT alignment first averages each night's minutes into 1%-of-SPT bins, then averages nights; 1% is not one minute.

In [ ]:
for alignment, xlabel in [("onset", "Minutes from SPT start"), ("wake", "Minutes before SPT end (negative)"), ("percent", "SPT (%)")]:
    fig, axes = plt.subplots(3, 2, figsize=(14, 10), constrained_layout=True)
    for ax, (metric, label) in zip(axes.flat, LABELS.items()):
        individual, group = aligned_profiles(minutes, metric, alignment)
        ax.plot(individual.index, individual, color="#167D8D", alpha=.08, linewidth=.6)
        ax.plot(group.index, group["mean"], color="#125B69", linewidth=1.4)
        ax.fill_between(group.index, group["mean"]-group.sd, group["mean"]+group.sd, color="#167D8D", alpha=.18)
        ax.set(xlabel=xlabel, ylabel=label)
        axes[2, 0 if metric in ["hr_mean_bpm", "quiet_hr_bpm"] else 1].plot(group.index, group.n_nights, label=label)
    for ax in axes[2]:
        ax.set(xlabel=xlabel, ylabel="Contributing nights", ylim=(0, len(nights)+1)); ax.legend(fontsize=8)
    finish("aligned_" + alignment)


## One-minute versus five-minute RMSSD within each night

Each panel preserves missing windows. Five-minute values represent ±2.5 minutes around the centre, so they smooth local changes and require longer uninterrupted quiet periods. Different availability is expected. The two sets of windows are not pooled together.

In [ ]:
fig, axes = plt.subplots(int(np.ceil(len(nights)/3)), 3, figsize=(16, 3*np.ceil(len(nights)/3)), squeeze=False, constrained_layout=True)
for ax, night in zip(axes.flat, nights.index):
    a, b = minutes.loc[minutes.night == night], five.loc[five.night == night]
    ax.plot(a.elapsed_min, a.rmssd_ms, color="#aaa", linewidth=.7, label="1 minute")
    ax.plot(b.elapsed_min, b.rmssd_ms, color="#B52E55", linewidth=1.1, label="5 minutes / 1-min step")
    ax.set(title=night[:10], xlabel="Minutes from SPT start", ylabel="RMSSD (ms)")
for ax in list(axes.flat)[len(nights):]: ax.set_visible(False)
axes.flat[0].legend(fontsize=7)
finish("rmssd_window_comparison")


## Early-to-late changes

First/last-hour differences require at least 15 valid minutes in each hour (counts are exported). They are descriptive and use available quiet periods for HRV. The SPT-third table includes valid-minute counts so sparse coverage remains visible. No sleep stages are inferred.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True)
for ax, (metric, label) in zip(axes.flat, LABELS.items()):
    ax.axhline(0, color="gray", linewidth=.8)
    ax.plot(x, summary[metric + "_last_minus_first"], "o-")
    ax.set_ylabel("Last − first hour: " + label); ax.tick_params(axis="x", rotation=45)
finish("early_late_changes")
display(summary[[c for c in summary if any(tag in c for tag in ["_early_", "_middle_", "_late_", "_first_hour_", "_last_hour_"]) ]])


## Private exports and provenance

Tables, figures, and any executed notebook stay in the Git-ignored outputs directory. The source notebook has empty outputs. Settings retain original quiet-period/cleaning provenance per night; one-minute QC includes every rejected window.

In [ ]:
for name in ["minutes", "five_minutes", "nights", "summary"]:
    result[name].to_csv(OUTPUT_DIR / (name + ".csv"), index=name in ["nights", "summary"])
for alignment in ["onset", "wake", "percent"]:
    for metric in LABELS:
        _, table = aligned_profiles(minutes, metric, alignment)
        table.to_csv(OUTPUT_DIR / f"aligned_{alignment}_{metric}.csv")
(OUTPUT_DIR / "settings.json").write_text(json.dumps({
    "night_ids": list(nights.index), "naive_timezone": NAIVE_TIMEZONE,
    "max_interpolated_fraction": MAX_INTERPOLATED_FRACTION,
    "windows_s": [60, 300], "step_s": 60, "hr_min_coverage": .9,
    "hr_max_gap_s": 3, "bbi_max_gap_s": 5, "bbi_duration_fraction": [.9, 1.1],
    "source_settings": result["source_settings"],
}, indent=2), encoding="utf-8")
print(f"Local outputs: {OUTPUT_DIR}")
